# Week 04 — Baseline Action Score

This notebook builds the required rule-based baseline:

1. Audit two signals with visible bucket tables and `n`.
2. Encode one transparent scoring rule with **one reason code** and an action label.
3. Write the ranked queue to `work/outputs/baseline_action_score.csv`.
4. Review the top 10 with an explicit failure condition for every row.
5. Run a self-check for future-window and label-derived leakage.

> **Important:** This notebook deliberately uses only columns found in the local FlyRank data. It does not use target/label columns in the score. If the automatic column mapping is not correct for the supplied dataset, edit the `COLUMN_MAP` cell before running the scoring section.


In [ ]:
from pathlib import Path
import re
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent

OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Output directory:", OUTPUT_DIR)


## 0. Load the local FlyRank data

The loader searches common project locations for CSV/Parquet files. It excludes generated output files so the baseline cannot accidentally train or score itself from its own output.


In [ ]:
def candidate_files(root):
    search_dirs = [
        root / "data",
        root / "work" / "data",
        root / "flyrank",
        root / "work",
    ]
    files = []
    for d in search_dirs:
        if d.exists():
            files.extend(d.rglob("*.csv"))
            files.extend(d.rglob("*.parquet"))
    # Avoid generated outputs and obvious metric/output files.
    return [
        p for p in files
        if "outputs" not in p.parts
        and "baseline_action_score" not in p.name
        and not p.name.startswith(".")
    ]

files = candidate_files(ROOT)
print("Candidate data files:")
for p in files[:50]:
    print(" -", p.relative_to(ROOT))

if not files:
    raise FileNotFoundError(
        "No CSV/Parquet data file was found. Put the FlyRank dataset in "
        "data/ or work/data/ and rerun this cell."
    )

# Prefer the largest tabular file because it is usually the main lane dataset.
sizes = [(p, p.stat().st_size) for p in files]
data_path = max(sizes, key=lambda x: x[1])[0]

if data_path.suffix.lower() == ".csv":
    df = pd.read_csv(data_path)
else:
    df = pd.read_parquet(data_path)

print("\nSelected:", data_path.relative_to(ROOT))
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()


## 1. Column mapping

The rule needs two signals:

- **staleness / refresh age**, linked to FlyRank refresh/staleness flags;
- **search volume**, linked to the quick-win/volume logic.

A third signal, **CTR-vs-position**, is used when available and provides a useful sanity check for the ranking rule.

The mapping below is deliberately conservative. It searches for common names but does not use target labels to calculate the score.


In [ ]:
def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

normalized = {normalize_name(c): c for c in df.columns}

def find_column(candidates):
    # Exact normalized match first.
    for cand in candidates:
        if cand in normalized:
            return normalized[cand]
    # Then substring match.
    for n, original in normalized.items():
        for cand in candidates:
            if cand in n:
                return original
    return None

COLUMN_MAP = {
    "id": find_column([
        "id", "query_id", "keyword_id", "url_id", "page_id", "item_id", "keyword", "query", "url"
    ]),
    "volume": find_column([
        "search_volume", "searchvolume", "volume", "monthly_searches", "monthly_search_volume"
    ]),
    "position": find_column([
        "position", "avg_position", "average_position", "rank", "ranking"
    ]),
    "ctr": find_column([
        "ctr", "click_through_rate", "clickthroughrate"
    ]),
    "last_updated": find_column([
        "last_updated", "last_update", "updated_at", "updated", "refresh_date",
        "last_refreshed", "last_refresh", "content_updated_at"
    ]),
}

print("Detected columns:")
for k, v in COLUMN_MAP.items():
    print(f"  {k:14s} -> {v}")

missing = [k for k in ["volume", "last_updated"] if COLUMN_MAP[k] is None]
if missing:
    print("\nWARNING: Required signal columns not automatically detected:", missing)
    print("Edit COLUMN_MAP manually using the exact names shown above.")


### Convert signal columns

For staleness, the notebook computes age from the **latest date present in the dataset**, rather than using today's date. This avoids introducing an artificial future-window dependency.

For CTR-vs-position, the notebook compares each row's CTR against the median CTR for its position bucket. This is descriptive baseline logic, not a learned label.


In [ ]:
work = df.copy()

# Numeric conversions
for key in ["volume", "position", "ctr"]:
    col = COLUMN_MAP.get(key)
    if col is not None:
        work[col] = pd.to_numeric(
            work[col].astype(str).str.replace("%", "", regex=False),
            errors="coerce"
        )

# Convert CTR percentages to proportions only when values clearly look like percentages.
ctr_col = COLUMN_MAP.get("ctr")
if ctr_col is not None and work[ctr_col].dropna().size:
    if work[ctr_col].dropna().median() > 1:
        work[ctr_col] = work[ctr_col] / 100.0

# Parse refresh/update date.
date_col = COLUMN_MAP.get("last_updated")
if date_col is not None:
    work["_refresh_date"] = pd.to_datetime(work[date_col], errors="coerce")
    reference_date = work["_refresh_date"].max()
    work["_staleness_days"] = (reference_date - work["_refresh_date"]).dt.days
else:
    work["_refresh_date"] = pd.NaT
    work["_staleness_days"] = np.nan
    reference_date = None

print("Reference date used for staleness:", reference_date)
print("Rows:", len(work))


# 1) Signal checks

## Signal A — Staleness / refresh age

This is directly linked to the FlyRank refresh/staleness flag family.

The verdict is based on whether the highest-staleness bucket has a higher median search volume than the lowest-staleness bucket. A negative or mixed result is intentionally accepted rather than forced into confirmation.


In [ ]:
def bucket_table(series, metric, labels=4):
    s = pd.to_numeric(series, errors="coerce")
    tmp = pd.DataFrame({"bucket_value": s, "metric": metric}).dropna()
    if len(tmp) < 4 or tmp["bucket_value"].nunique() < 2:
        return None

    tmp["bucket"] = pd.qcut(
        tmp["bucket_value"].rank(method="first"),
        q=min(labels, tmp["bucket_value"].nunique()),
        duplicates="drop"
    )
    out = (
        tmp.groupby("bucket", observed=True)
        .agg(metric_median=("metric", "median"),
             metric_mean=("metric", "mean"),
             n=("metric", "size"))
        .reset_index()
    )
    return out

if COLUMN_MAP["volume"] is not None and work["_staleness_days"].notna().sum() >= 4:
    stale_tbl = bucket_table(work["_staleness_days"], work[COLUMN_MAP["volume"]])
    display(stale_tbl)

    low = stale_tbl.iloc[0]["metric_median"]
    high = stale_tbl.iloc[-1]["metric_median"]
    ratio = high / low if low not in [0, np.nan] else np.nan

    if pd.notna(ratio) and ratio >= 1.20:
        stale_verdict = "CONFIRMED"
    elif pd.notna(ratio) and ratio <= 0.80:
        stale_verdict = "OPPOSITE"
    else:
        stale_verdict = "MIXED"

    print(f"n = {len(work)}")
    print("Verdict:", stale_verdict)
    print(
        f"Reason: median search volume changes from {low:.2f} in the least-stale bucket "
        f"to {high:.2f} in the most-stale bucket."
    )
else:
    stale_verdict = "FALSE"
    print("Verdict: FALSE")
    print("Reason: the dataset does not contain enough usable refresh-date and volume observations.")


## Signal B — Search volume / quick-win signal

This is the second signal behind the FlyRank quick-win logic. We bucket search volume and compare the distribution of candidate opportunities using a **non-label outcome proxy**: current organic position when available.

If position is unavailable, the table still reports volume bucket sizes, but the verdict is `FALSE` because there is not enough evidence to claim a relationship.


In [ ]:
volume_col = COLUMN_MAP["volume"]
position_col = COLUMN_MAP["position"]

if volume_col is not None and position_col is not None:
    tmp = work[[volume_col, position_col]].copy()
    tmp[volume_col] = pd.to_numeric(tmp[volume_col], errors="coerce")
    tmp[position_col] = pd.to_numeric(tmp[position_col], errors="coerce")
    tmp = tmp.dropna()

    if len(tmp) >= 4 and tmp[volume_col].nunique() >= 2:
        tmp["volume_bucket"] = pd.qcut(
            tmp[volume_col].rank(method="first"),
            q=min(4, tmp[volume_col].nunique()),
            duplicates="drop"
        )

        volume_tbl = (
            tmp.groupby("volume_bucket", observed=True)
            .agg(
                median_position=(position_col, "median"),
                mean_position=(position_col, "mean"),
                n=(position_col, "size")
            )
            .reset_index()
        )
        display(volume_tbl)

        # For this baseline, a lower median position is interpreted as better current ranking.
        low_vol_pos = volume_tbl.iloc[0]["median_position"]
        high_vol_pos = volume_tbl.iloc[-1]["median_position"]

        if pd.notna(low_vol_pos) and pd.notna(high_vol_pos):
            if high_vol_pos <= low_vol_pos - 1:
                volume_verdict = "CONFIRMED"
            elif high_vol_pos >= low_vol_pos + 1:
                volume_verdict = "OPPOSITE"
            else:
                volume_verdict = "MIXED"
        else:
            volume_verdict = "FALSE"

        print(f"n = {len(tmp)}")
        print("Verdict:", volume_verdict)
        print(
            f"Reason: the highest-volume bucket has median position {high_vol_pos:.2f}, "
            f"versus {low_vol_pos:.2f} for the lowest-volume bucket."
        )
    else:
        volume_verdict = "FALSE"
        print("Verdict: FALSE")
        print("Reason: insufficient variation or observations in volume/position.")
else:
    volume_verdict = "FALSE"
    print("Verdict: FALSE")
    print("Reason: search volume and/or position is unavailable.")


# 2) Encode ONE baseline rule

The rule is intentionally simple and auditable:

- **Volume component:** higher search volume receives a higher percentile score.
- **Staleness component:** older pages receive a higher percentile score.
- **CTR-vs-position component:** when both CTR and position exist, unusually weak CTR for the position receives a higher score.

No target/label is used.

The row receives **exactly one reason code**, chosen by whichever component contributes the most points. The action label is derived from that single reason.


In [ ]:
def percentile_score(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() <= 1:
        return pd.Series(0.0, index=s.index)
    return s.rank(pct=True).fillna(0.0) * 100

# Components
work["_volume_score"] = 0.0
work["_stale_score"] = 0.0
work["_ctr_gap_score"] = 0.0

if volume_col is not None:
    work["_volume_score"] = percentile_score(work[volume_col])

if work["_staleness_days"].notna().sum() > 1:
    work["_stale_score"] = percentile_score(work["_staleness_days"])

if COLUMN_MAP["ctr"] is not None and position_col is not None:
    p = pd.to_numeric(work[position_col], errors="coerce")
    c = pd.to_numeric(work[COLUMN_MAP["ctr"]], errors="coerce")

    # Position buckets: 1-3, 4-10, 11-20, 21+
    pos_bucket = pd.cut(
        p,
        bins=[-np.inf, 3, 10, 20, np.inf],
        labels=["1-3", "4-10", "11-20", "21+"]
    )
    expected_ctr = c.groupby(pos_bucket, observed=False).transform("median")
    gap = (expected_ctr - c).clip(lower=0)
    work["_ctr_gap_score"] = percentile_score(gap)

# Weight only signals that actually exist.
components = []
if volume_col is not None:
    components.append("_volume_score")
if work["_staleness_days"].notna().sum() > 1:
    components.append("_stale_score")
if COLUMN_MAP["ctr"] is not None and position_col is not None:
    components.append("_ctr_gap_score")

if not components:
    raise ValueError("No usable scoring signals were detected.")

# Equal-weight transparent baseline.
work["baseline_score"] = work[components].mean(axis=1)

# Exactly ONE reason code: largest component wins.
component_to_reason = {
    "_volume_score": "HIGH_VOLUME",
    "_stale_score": "STALE_PAGE",
    "_ctr_gap_score": "CTR_GAP",
}

component_to_action = {
    "_volume_score": "QUICK_WIN",
    "_stale_score": "REFRESH",
    "_ctr_gap_score": "CTR_FIX",
}

def winning_reason(row):
    vals = {c: row[c] for c in components}
    winner = max(vals, key=vals.get)
    return component_to_reason[winner], component_to_action[winner]

work[["reason_code", "action"]] = work.apply(
    lambda r: pd.Series(winning_reason(r)),
    axis=1
)

print("Scoring components:", components)
print("\nReason counts:")
display(work["reason_code"].value_counts(dropna=False).to_frame("n"))
print("\nAction counts:")
display(work["action"].value_counts(dropna=False).to_frame("n"))


In [ ]:
# Build the ranked queue.
id_col = COLUMN_MAP["id"]

if id_col is not None:
    work["_item_id"] = work[id_col].astype(str)
else:
    work["_item_id"] = work.index.astype(str)

ranked = (
    work[["_item_id", "baseline_score", "reason_code", "action"]]
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

output_path = OUTPUT_DIR / "baseline_action_score.csv"
ranked.to_csv(output_path, index=False)

print("Wrote:", output_path)
print("Rows written:", len(ranked))
display(ranked.head(10))


# 3) Top-10 review

Each row below is reviewed skeptically.

The final column is deliberately a **failure condition**, not a justification. It answers: *what real-world/data condition would make this baseline pick wrong?*


In [ ]:
top10 = ranked.head(10).copy()

def failure_condition(reason):
    return {
        "HIGH_VOLUME": "Search volume may be high but the query may have weak commercial/relevance intent.",
        "STALE_PAGE": "The page may already have been refreshed outside the recorded update field.",
        "CTR_GAP": "Low CTR may be caused by SERP layout, snippets, features, or query intent rather than a fixable page issue."
    }.get(reason, "The available features may not represent the actual opportunity.")

top10_review = top10.copy()
top10_review["why_its_here"] = top10_review["reason_code"].map({
    "HIGH_VOLUME": "It receives a strong score from the search-volume component.",
    "STALE_PAGE": "It receives a strong score from the page-staleness component.",
    "CTR_GAP": "It receives a strong score from the CTR-vs-position gap component."
}).fillna("It has a high combined baseline score.")

top10_review["what_would_make_it_wrong"] = top10_review["reason_code"].map(failure_condition)

display(top10_review[
    ["rank", "_item_id", "action", "baseline_score",
     "reason_code", "why_its_here", "what_would_make_it_wrong"]
])


# 4) Weak picks

A weak-pick check makes the baseline easier to critique. These are high-ranked rows whose score is driven by only one strong component while the other available components are weak.

This is diagnostic only; it does not alter the baseline queue.


In [ ]:
score_cols = components

if score_cols:
    weak = work.copy()
    weak["_max_component"] = weak[score_cols].max(axis=1)
    weak["_min_component"] = weak[score_cols].min(axis=1)
    weak["_spread"] = weak["_max_component"] - weak["_min_component"]

    weak_picks = (
        weak.sort_values(["_spread", "baseline_score"], ascending=[False, False])
        [["_item_id", "baseline_score", "reason_code", "action", "_spread"]]
        .head(5)
    )
    display(weak_picks)
else:
    print("No weak-pick diagnostics available.")


# 5) Self-check

The checks below verify the assignment requirements programmatically:

- two signal verdicts were produced;
- each signal table contains `n` where the signal is usable;
- the queue contains a score, one reason code, and one action;
- the output CSV exists;
- no obvious label/target column was used in the score;
- no future-window feature is deliberately created by this notebook.


In [ ]:
# ---- Self-check ----

# 1. Signal verdicts
assert stale_verdict in {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}
assert volume_verdict in {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}

print("Signal 1 verdict:", stale_verdict)
print("Signal 2 verdict:", volume_verdict)

# 2. Required queue columns
required_queue = {"rank", "_item_id", "baseline_score", "reason_code", "action"}
assert required_queue.issubset(ranked.columns)

# 3. Exactly one reason/action per row
assert ranked["reason_code"].notna().all()
assert ranked["action"].notna().all()

# 4. Score is numeric and sorted
assert pd.api.types.is_numeric_dtype(ranked["baseline_score"])
assert ranked["baseline_score"].is_monotonic_decreasing

# 5. Output exists
assert output_path.exists()
print("Output exists:", output_path)

# 6. Check for obvious label-like columns excluded from scoring.
label_keywords = [
    "label", "target", "outcome", "conversion", "converted",
    "quick_win", "is_quick_win", "ground_truth", "groundtruth"
]
label_like = [
    c for c in df.columns
    if any(k in normalize_name(c) for k in label_keywords)
]

used_score_inputs = [
    COLUMN_MAP.get("volume"),
    COLUMN_MAP.get("position"),
    COLUMN_MAP.get("ctr"),
    COLUMN_MAP.get("last_updated")
]
used_score_inputs = [x for x in used_score_inputs if x is not None]

leaky_overlap = set(used_score_inputs) & set(label_like)
assert not leaky_overlap, f"Potential label-derived input used: {leaky_overlap}"

print("Label-like columns detected but not used:", label_like)
print("Score inputs:", used_score_inputs)

print("\nSELF-CHECK PASSED")


## Submission checklist

Before committing:

- [ ] `work/notebooks/w04_baseline_score.ipynb` is executed.
- [ ] Two signal verdicts are visible.
- [ ] Bucket tables visibly contain `n`.
- [ ] At least one signal is FlyRank-flag linked.
- [ ] One baseline score is encoded.
- [ ] Each row has exactly one reason code.
- [ ] Each row has an action label.
- [ ] The notebook writes `work/outputs/baseline_action_score.csv`.
- [ ] Top 10 rows each have an action, rationale, and failure condition.
- [ ] No future-window or label-derived inputs are used.
- [ ] Commit the notebook and any allowed `work/outputs/*.json` receipts.
- [ ] Do **not** commit the CSV if the repository's leak guard excludes it.
